# Grafana, without the monitoring stack

The no-LGTM variant. Grafana **is** the monitoring stack's UI, so when it is down its own
dashboards cannot tell you -- and neither can [grafana-health.ipynb](grafana-health.ipynb),
which reads Grafana's metrics out of Mimir and so depends on Grafana's scrape, Alloy, and Mimir
all working. This notebook uses **only instruments outside the stack**: the Kubernetes API, and
direct HTTP to Grafana itself.

Grafana appears here strictly as the **system under test**, never as an instrument. Nothing is
read through Mimir, Loki, or a dashboard.

The sharpest check is `/api/health`, which needs no authentication and no metrics pipeline:

```json
{"database": "ok", "version": "12.3.1", "commit": "..."}
```

`database: ok` means Grafana is up *and* its SQLite store on the PVC is readable -- which is the
failure this deployment is actually exposed to, being a single instance with persistence enabled
and no HA. `/metrics` is probed too, because "Grafana is fine but the scrape is broken" and
"Grafana is down" look identical from Mimir and completely different from here. That probe
checks the endpoint serves Prometheus exposition, not that it contains any particular metric:
the capture layer keeps only the first 4000 bytes and families are emitted alphabetically, so
`grafana_*` sits past the cut. Family-level checks belong in the health notebook.

**Scope note:** deliberately small. It answers "is Grafana up, serving, and backed by a readable
database", not "why".

---

**Using this notebook to debug something? Improve it before you finish.** These notebooks exist
to move knowledge out of one agent's transcript, so a query, threshold or cross-check that turned
out to matter here belongs *in these cells* before the session ends -- not left behind in a chat
log for nobody. The motivating case is
[#656](https://github.com/symmatree/tiles/issues/656): the alloy-singleton crashloop was caught
only because an earlier session had baked its component-health check into `alloy-health.ipynb`,
and the next reader inherited it. See [docs/notebooks.md](../docs/notebooks.md).


In [ ]:
# No Mimir, no Loki, no Grafana datasource -- by design.
grafana_url = "http://grafana.grafana.svc"            # in-cluster Service
namespace = "grafana"
deployment_label = "app.kubernetes.io/name=grafana"
expect_containers = 3        # grafana + dashboard sidecar + datasource sidecar
ingress_host = ""            # optional: set to probe the external URL from in-cluster
restart_warn = 3             # chosen: restarts since pod creation worth a mention
capture_file = ""
output_dir = ""

In [ ]:
import json
import numpy as np
from pathlib import Path
import nb_capture as nbc

NOTEBOOK = 'grafana-nomon'
cap = nbc.Capture(replay_file=capture_file, grafana_url=grafana_url, namespace=namespace,
                  deployment_label=deployment_label, expect_containers=expect_containers,
                  ingress_host=ingress_host, restart_warn=restart_warn)
m = cap.meta
grafana_url, namespace = m['grafana_url'], m['namespace']
deployment_label, expect_containers = m['deployment_label'], m['expect_containers']
ingress_host, restart_warn = m['ingress_host'], m['restart_warn']
kube = nbc.Kube(cap)

print(f"grafana: {grafana_url}  namespace: {namespace}  "
      f"{'REPLAY of ' + cap.run_at if cap.replay else 'live'}")

In [ ]:
# --- ASSUMPTIONS: the kube API is the one instrument this notebook cannot do without ---
try:
    pods = kube.get('kube: pods', 'pods', '-n', namespace, '-l', deployment_label)
    kube_ok = True
except Exception as e:  # noqa: BLE001
    pods, kube_ok = {'items': []}, False
    kube_err = str(e)[:200]

print("=== ASSUMPTIONS ===")
print(f"  kube API   {'ok' if kube_ok else 'UNAVAILABLE: ' + kube_err}")
if not kube_ok:
    raise RuntimeError("unassessable: no kube API access -- this notebook has no other instrument")
print(f"  pods found {len(pods.get('items', []))}")

In [ ]:
# --- CAPTURE ---
events = kube.get('kube: events', 'events', '-n', namespace)
pvcs = kube.get('kube: pvcs', 'persistentvolumeclaims', '-n', namespace)
svcs = kube.get('kube: services', 'services', '-n', namespace)

# The system under test, probed directly. No auth on either endpoint.
health = cap.http('probe: /api/health', f'{grafana_url}/api/health', timeout=20)
metrics = cap.http('probe: /metrics', f'{grafana_url}/metrics', timeout=30)
ingress_probe = (cap.http('probe: ingress', f'https://{ingress_host}/api/health', timeout=20)
                 if ingress_host else None)
print(f"captured {len(cap.data['calls'])} raw observations")

## Is Grafana serving, and is its database readable?

In [ ]:
hj = health.get('json') or {}
hstatus = health.get('status')
print(f"/api/health  HTTP {hstatus}   {json.dumps(hj) if hj else health.get('error') or health.get('text','')[:200]}")
db_ok = hj.get('database') == 'ok'
print(f"  database: {hj.get('database', 'UNKNOWN')}   version: {hj.get('version', '?')}   "
      f"commit: {str(hj.get('commit', '?'))[:12]}")

mstatus = metrics.get('status')
mtext = metrics.get('text', '') or ''
# nb_capture truncates recorded bodies at 4000 bytes. client_golang emits families in
# alphabetical order, so this prefix is go_*/process_* and the grafana_* families sit beyond
# it -- their absence HERE says nothing. The honest check is therefore "the endpoint serves
# Prometheus exposition", not "it contains metric X"; family-level checks belong in
# grafana-health.ipynb, which reads them after they have been through the pipeline.
sample_lines = [ln for ln in mtext.splitlines() if ln and not ln.startswith('#')]
names = {ln.split('{')[0].split(' ')[0] for ln in sample_lines}
exposition_ok = mstatus == 200 and '# HELP' in mtext and len(sample_lines) > 0
print(f"\n/metrics     HTTP {mstatus}   {len(mtext)} bytes captured (truncated by the capture layer)")
print(f"  parses as Prometheus exposition: {exposition_ok}"
      f"   ({len(names)} metric names in the captured prefix, e.g. {sorted(names)[:3]})")
if exposition_ok:
    print("  -> Grafana is serving its metrics endpoint. If grafana-health.ipynb reports the")
    print("     metrics stale, the break is in the ServiceMonitor/Alloy/Mimir path, not here.")

if ingress_probe is not None:
    ij = ingress_probe.get('json') or {}
    print(f"\ningress https://{ingress_host}/api/health  HTTP {ingress_probe.get('status')}"
          f"   {json.dumps(ij) if ij else ingress_probe.get('error','')[:120]}")

## Pod, container and storage state

A single-instance Grafana with persistence: the pod and its PVC are the whole story. All three
containers must be ready -- a dead sidecar means dashboards or datasources silently stop
updating while Grafana itself looks perfectly healthy.

In [ ]:
items = pods.get('items', [])
print(f"pods in {namespace} matching {deployment_label}: {len(items)}")
restart_total = 0
containers_ready = 0
for p in items:
    st = p.get('status', {})
    meta = p.get('metadata', {})
    cs = st.get('containerStatuses', []) or []
    ready = sum(1 for c in cs if c.get('ready'))
    containers_ready = max(containers_ready, ready)
    print(f"  {meta.get('name')}  phase={st.get('phase')}  ready={ready}/{len(cs)}  "
          f"node={p.get('spec', {}).get('nodeName')}")
    for c in cs:
        r = c.get('restartCount', 0)
        restart_total += r
        last = (c.get('lastState') or {}).get('terminated') or {}
        note = ''
        if last:
            note = (f"   last exit: {last.get('reason')} code={last.get('exitCode')}"
                    f" at {last.get('finishedAt')}")
        print(f"      {c.get('name'):<26} ready={str(c.get('ready')):<5} restarts={r}{note}")

print(f"\ncontainers ready: {containers_ready}/{expect_containers} expected")
print("\nPVCs:")
for v in pvcs.get('items', []):
    s = v.get('status', {})
    print(f"  {v['metadata']['name']:<24} {s.get('phase')}  "
          f"{s.get('capacity', {}).get('storage', '?')}  sc={v['spec'].get('storageClassName')}")

print("\nServices:")
for s in svcs.get('items', []):
    print(f"  {s['metadata']['name']:<34} {s['spec'].get('type')}  "
          f"ports={[p.get('port') for p in s['spec'].get('ports', [])]}")

ev = events.get('items', [])
warn_ev = [e for e in ev if e.get('type') != 'Normal']
print(f"\nkube events in the namespace: {len(ev)} ({len(warn_ev)} non-Normal)."
      f"  Events are TTL'd (~1h), so this is an during/just-after-incident view, not history.")
for e in warn_ev[:10]:
    print(f"  {e.get('lastTimestamp')} {e.get('type')} {e.get('reason')}: {str(e.get('message'))[:110]}")

## Summary

In [ ]:
findings = []
if hstatus != 200:
    findings.append(f"/api/health returned HTTP {hstatus} -- Grafana is not serving "
                    f"({health.get('error') or str(health.get('text',''))[:120]})")
elif not db_ok:
    findings.append(f"/api/health reports database={hj.get('database')!r} -- Grafana is up but its "
                    f"store is not readable (single instance on a PVC; check the PVC and the node)")
if not exposition_ok:
    findings.append(f"/metrics did not return usable Prometheus exposition (HTTP "
                    f"{metrics.get('status')}) -- grafana-health.ipynb cannot work either")
for p in items:
    st = p.get('status', {})
    cs = st.get('containerStatuses', []) or []
    if st.get('phase') != 'Running':
        findings.append(f"pod {p['metadata']['name']} phase={st.get('phase')}")
    for c in cs:
        if not c.get('ready'):
            findings.append(f"container {c.get('name')} NOT ready in {p['metadata']['name']}"
                            + (" -- dashboards/datasources will stop updating silently"
                               if 'sc-' in str(c.get('name')) else ""))
        if c.get('restartCount', 0) >= restart_warn:
            last = (c.get('lastState') or {}).get('terminated') or {}
            findings.append(f"container {c.get('name')} has {c['restartCount']} restarts"
                            + (f", last {last.get('reason')}" if last else ""))
if containers_ready < expect_containers:
    findings.append(f"only {containers_ready}/{expect_containers} containers ready")
for v in pvcs.get('items', []):
    if v.get('status', {}).get('phase') != 'Bound':
        findings.append(f"PVC {v['metadata']['name']} is {v.get('status', {}).get('phase')}")
if warn_ev:
    findings.append(f"{len(warn_ev)} non-Normal kube events in the last ~hour")
if ingress_probe is not None and ingress_probe.get('status') != 200:
    findings.append(f"ingress probe returned {ingress_probe.get('status')} "
                    f"-- in-cluster is fine but the external path may not be")

print("=== VERDICT (system under test) ===")
print(f"  serving: {'yes' if hstatus == 200 else 'NO'}    "
      f"database: {hj.get('database', 'unknown')}    "
      f"metrics endpoint: {'up' if metrics.get('status') == 200 else 'DOWN'}    "
      f"containers ready: {containers_ready}/{expect_containers}")
print("\n=== FINDINGS ===")
for f_ in findings:
    print(f"  - {f_}")
if not findings:
    print("  none: Grafana is serving, its database is readable, all containers are ready,")
    print("  and its PVC is bound -- checked without using the monitoring stack at all.")

In [ ]:
# --- AGENT-READABLE STATS ---
agent_stats = {
    'notebook': f'{NOTEBOOK}.ipynb', 'run_at': cap.run_at, 'mode': cap.mode,
    'assumptions': {'status': 'ok' if kube_ok else 'unassessable', 'kube_available': kube_ok,
                    'instruments': ['kube API', 'direct HTTP to Grafana'],
                    'uses_monitoring_stack': False},
    'findings': findings,
    'system_under_test': {
        'health_http_status': health.get('status'),
        'database': hj.get('database'), 'version': hj.get('version'),
        'commit': hj.get('commit'),
        'metrics_http_status': metrics.get('status'),
        'metrics_exposition_ok': exposition_ok,
        'containers_ready': containers_ready, 'containers_expected': expect_containers,
        'restarts_total': restart_total,
        'ingress_http_status': (ingress_probe or {}).get('status') if ingress_probe else None,
    },
    'pods': [{'name': p['metadata']['name'], 'phase': p.get('status', {}).get('phase'),
              'node': p.get('spec', {}).get('nodeName'),
              'containers': [{'name': c.get('name'), 'ready': c.get('ready'),
                              'restarts': c.get('restartCount', 0)}
                             for c in (p.get('status', {}).get('containerStatuses') or [])]}
             for p in items],
    'pvcs': {v['metadata']['name']: v.get('status', {}).get('phase') for v in pvcs.get('items', [])},
    'events_non_normal': len(warn_ev),
}
print(json.dumps(agent_stats, indent=1))

if output_dir:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cap.save(out / f'{NOTEBOOK}-capture.json')
    (out / f'{NOTEBOOK}-stats.json').write_text(json.dumps(agent_stats, indent=1))
    print(f"\nwrote {out}/{NOTEBOOK}-capture.json and -stats.json")